## Day 4: Data Transformation, Mapping, Grouping & Aggregation

Welcome to Day 4!

Yesterday you learned **data cleaning** — handling missing values, duplicates, type conversion, and string operations.

Today you learn how to **transform data** and **extract insights**:

**Think:**

So far you can:
- ✅ Load data
- ✅ Inspect data
- ✅ Clean data

Today you'll learn how to **extract insights** and **create new features**.

This is where Pandas becomes a true data analysis tool.

**Example:**

Instead of looking at every student individually, you can answer questions like:
- Which department has the highest average score?
- Which department has the highest salary?
- Which city has the most customers?
- Which product sells the most?

These questions are answered with `groupby()`.

## 📈 Roadmap Progress

| Day | Topic | Status |
|-----|-------|--------|
| ✅ Day 1 | Pandas Foundations — Series, DataFrames & Data Loading | Complete |
| ✅ Day 2 | Data Selection, Filtering, Indexing & Sorting | Complete |
| ✅ Day 3 | Data Cleaning — Missing Values, Duplicates, Type Conversion & String Operations | Complete |
| 🚀 Day 4 | Data Transformation, Mapping, Grouping & Aggregation | **In Progress** |
| ⏳ Day 5 | Combining Data | Pending |
| ⏳ Day 6 | Advanced Pandas | Pending |
| ⏳ Day 7 | Real-World EDA & ML Workflow | Pending |


## 🎯 Learning Objectives

By the end of Day 4, you will be able to:

- Transform columns using `map()`
- Create new features using `apply()`
- Use anonymous (lambda) functions
- Understand when **not** to use `apply()`
- Group data with `groupby()`
- Perform aggregations with `agg()`
- Transform grouped data using `transform()`
- Filter groups with `filter()`
- Create pivot tables
- Create cross-tabulations
- Perform feature engineering like a Data Scientist


## Why This Day Is Important

Everything you've learned so far lets you:

1. Load data
2. Inspect data
3. Clean data

Today you'll learn how to **extract insights** 👇 This is where Pandas becomes a true data analysis tool.


In [33]:
import pandas as pd
import numpy as np

students = {
    "Name": [
        "Alice",
        "Bob",
        "Charlie",
        "David",
        "Eve",
        "Frank",
        "Grace"
    ],
    "Department": [
        "CS",
        "IT",
        "CS",
        "SE",
        "IT",
        "CS",
        "SE"
    ],
    "Math": [80,70,90,60,95,75,88],
    "Python": [90,85,95,70,100,80,85],
    "AI": [88,75,92,65,98,78,87]
}

df = pd.DataFrame(students)

print(df)

      Name Department  Math  Python  AI
0    Alice         CS    80      90  88
1      Bob         IT    70      85  75
2  Charlie         CS    90      95  92
3    David         SE    60      70  65
4      Eve         IT    95     100  98
5    Frank         CS    75      80  78
6    Grace         SE    88      85  87


---

## PART 1 — `map()`

### What is `map()`?

`map()` transforms values in **one Series**.

**Signature**

```python
Series.map(arg)
```

### Example 1 — Convert department abbreviations

> We keep the original `Department` column (abbreviated) for later `groupby` work, and store the full names in a new column.


In [34]:
department_map = {
    "CS": "Computer Science",
    "IT": "Information Technology",
    "SE": "Software Engineering"
}

df["Department_Name"] = df["Department"].map(department_map)

print(df)

      Name Department  Math  Python  AI         Department_Name
0    Alice         CS    80      90  88        Computer Science
1      Bob         IT    70      85  75  Information Technology
2  Charlie         CS    90      95  92        Computer Science
3    David         SE    60      70  65    Software Engineering
4      Eve         IT    95     100  98  Information Technology
5    Frank         CS    75      80  78        Computer Science
6    Grace         SE    88      85  87    Software Engineering


### Example 2 — Convert grades

You can also `map()` a `Series` of its own values.


In [35]:
grades = pd.Series(
    ["A", "B", "C", "A"]
)

mapping = {
    "A": 4,
    "B": 3,
    "C": 2
}

# print(grades)
print(grades.map(mapping))

0    4
1    3
2    2
3    4
dtype: int64


### `map()` vs `replace()`

| | `map()` | `replace()` |
|--|---------|-------------|
| Works on | A **Series** only | A **Series** or **DataFrame** |
| Argument | A dict or a function | A dict, list, scalar, or regex |
| Unknown values | Become `NaN` | Remain **unchanged** |
| Partial replacement | Not supported | Supported (regex / substring) |

**Example:**


In [36]:
s = pd.Series(["CS", "IT", "LAW"])

mapping = {
    "CS": "Computer Science",
    "IT": "Information Technology"
}

print("map() result:")
print(s.map(mapping))
print()
print("replace() result:")
print(s.replace(mapping))

map() result:
0          Computer Science
1    Information Technology
2                       NaN
dtype: object

replace() result:
0          Computer Science
1    Information Technology
2                       LAW
dtype: object


Notice how `map()` turns `"LAW"` into `NaN`, while `replace()` leaves it untouched. Both work on a `Series` here — but unlike `map()`, `replace()` can also be applied to a whole `DataFrame`.


## PART 2 — `apply()`

### What is `apply()`?

`apply()` applies a function **to each value**.

- On a **DataFrame**: applies a function to each column (default) or each row.
- On a **Series**: applies a function to each value.

**Signature**

```python
DataFrame.apply(func, axis=0)   # axis=0 -> columns, axis=1 -> rows
```

### Column-wise (default `axis=0`)


In [37]:
print("Column means:")
print(df[["Math", "Python", "AI"]].apply(np.mean))

Column means:
Math      79.714286
Python    86.428571
AI        83.285714
dtype: float64


### Row-wise (`axis=1`)

Each row = one student.


In [38]:
df["Row_Mean"] = df[["Math", "Python", "AI"]].apply(np.mean, axis=1)
print(df)

      Name Department  Math  Python  AI         Department_Name   Row_Mean
0    Alice         CS    80      90  88        Computer Science  86.000000
1      Bob         IT    70      85  75  Information Technology  76.666667
2  Charlie         CS    90      95  92        Computer Science  92.333333
3    David         SE    60      70  65    Software Engineering  65.000000
4      Eve         IT    95     100  98  Information Technology  97.666667
5    Frank         CS    75      80  78        Computer Science  77.666667
6    Grace         SE    88      85  87    Software Engineering  86.666667


### Using Custom Functions

Instead of an anonymous function, define a named function:


In [39]:
def total_score(row):
    return row["Math"] + row["Python"] + row["AI"]

# Make a fresh copy for a clean demonstration
demo = df.copy()
demo["Total"] = demo.apply(total_score, axis=1)
print(demo[["Name", "Math", "Python", "AI", "Total"]])

      Name  Math  Python  AI  Total
0    Alice    80      90  88    258
1      Bob    70      85  75    230
2  Charlie    90      95  92    277
3    David    60      70  65    195
4      Eve    95     100  98    293
5    Frank    75      80  78    233
6    Grace    88      85  87    260


### Lambda Functions

Instead of

```python
def square(x):
    return x*x
```

Use

```python
lambda x: x*x
```

#### Example — squared scores


In [40]:
df["Math_Squared"] = df["Math"].apply(lambda x: x*x)
print(df[["Name", "Math", "Math_Squared"]])

      Name  Math  Math_Squared
0    Alice    80          6400
1      Bob    70          4900
2  Charlie    90          8100
3    David    60          3600
4      Eve    95          9025
5    Frank    75          5625
6    Grace    88          7744


#### Example — Pass / Fail


In [41]:
df["Pass"] = df["Math"].apply(lambda x: "Pass" if x >= 50 else "Fail")
print(df[["Name", "Math", "Pass"]])

      Name  Math  Pass
0    Alice    80  Pass
1      Bob    70  Pass
2  Charlie    90  Pass
3    David    60  Pass
4      Eve    95  Pass
5    Frank    75  Pass
6    Grace    88  Pass


### Function that returns a category — `grade()`


In [42]:
def grade(score):
    if score >= 90:
        return "A"
    elif score >= 80:
        return "B"
    elif score >= 70:
        return "C"
    return "D"

df["Grade"] = df["Python"].apply(grade)
print(df[["Name", "Python", "Grade"]])

      Name  Python Grade
0    Alice      90     A
1      Bob      85     B
2  Charlie      95     A
3    David      70     C
4      Eve     100     A
5    Frank      80     B
6    Grace      85     B


## PART 3 — `assign()`

Instead of

```python
df["Average"] = df[["Math", "Python", "AI"]].mean(axis=1)
```

You can use `assign()`. It is **chainable**, which makes it useful in pipelines.


In [43]:
df = df.assign(
    Average=df[["Math", "Python", "AI"]].mean(axis=1)
)
print(df[["Name", "Math", "Python", "AI", "Average"]])

      Name  Math  Python  AI    Average
0    Alice    80      90  88  86.000000
1      Bob    70      85  75  76.666667
2  Charlie    90      95  92  92.333333
3    David    60      70  65  65.000000
4      Eve    95     100  98  97.666667
5    Frank    75      80  78  77.666667
6    Grace    88      85  87  86.666667


The advantage of `assign()` shows up when chaining operations (we'll see this in later days).

## PART 4 — `groupby()`

This is the **heart of Pandas**.

Imagine our table split by `Department`:

```
Department = CS  ->  Alice, Charlie, Frank
Department = IT  ->  Bob, Eve
Department = SE  ->  David, Grace
```

When you run

```python
df.groupby("Department")
```

Pandas **internally creates groups** — but **nothing is calculated yet**. This is called a *"lazy"* operation.

The general pattern is:

```
df.groupby("Department").<aggregation>()
```

### Mean


In [44]:
print(df.groupby("Department").mean(numeric_only=True))

                 Math     Python    AI   Row_Mean  Math_Squared    Average
Department                                                                
CS          81.666667  88.333333  86.0  85.333333   6708.333333  85.333333
IT          82.500000  92.500000  86.5  87.166667   6962.500000  87.166667
SE          74.000000  77.500000  76.0  75.833333   5672.000000  75.833333


### Count
`count()` counts **non-null values** per column.


In [45]:
print(df.groupby("Department").count())

            Name  Math  Python  AI  Department_Name  Row_Mean  Math_Squared  \
Department                                                                    
CS             3     3       3   3                3         3             3   
IT             2     2       2   2                2         2             2   
SE             2     2       2   2                2         2             2   

            Pass  Grade  Average  
Department                        
CS             3      3        3  
IT             2      2        2  
SE             2      2        2  


### Size

`size()` counts **rows** (ignoring the column content entirely).


In [46]:
print(df.groupby("Department").size())

Department
CS    3
IT    2
SE    2
dtype: int64


**Difference:** `count()` returns a DataFrame with one entry per column; `size()` returns a single Series (one count per group).


In [47]:
print("count() is a DataFrame:")
print(type(df.groupby("Department").count()))
print()
print("size() is a Series:")
print(type(df.groupby("Department").size()))

count() is a DataFrame:
<class 'pandas.core.frame.DataFrame'>

size() is a Series:
<class 'pandas.core.series.Series'>


### Sum


In [48]:
print(df.groupby("Department").sum(numeric_only=True))

            Math  Python   AI    Row_Mean  Math_Squared     Average
Department                                                         
CS           245     265  258  256.000000         20125  256.000000
IT           165     185  173  174.333333         13925  174.333333
SE           148     155  152  151.666667         11344  151.666667


### Maximum


In [49]:
print(df.groupby("Department").max(numeric_only=True))

            Math  Python  AI   Row_Mean  Math_Squared    Average
Department                                                      
CS            90      95  92  92.333333          8100  92.333333
IT            95     100  98  97.666667          9025  97.666667
SE            88      85  87  86.666667          7744  86.666667


### Minimum


In [50]:
print(df.groupby("Department").min(numeric_only=True))

            Math  Python  AI   Row_Mean  Math_Squared    Average
Department                                                      
CS            75      80  78  77.666667          5625  77.666667
IT            70      85  75  76.666667          4900  76.666667
SE            60      70  65  65.000000          3600  65.000000


### Multiple Aggregations with `agg()`

You can apply **different aggregations to different columns**:


In [51]:
print(df.groupby("Department").agg({
    "Math": ["mean", "max", "min"],
    "Python": ["mean", "max"],
    "AI": ["mean"]
}))

                 Math             Python         AI
                 mean max min       mean  max  mean
Department                                         
CS          81.666667  90  75  88.333333   95  86.0
IT          82.500000  95  70  92.500000  100  86.5
SE          74.000000  88  60  77.500000   85  76.0


This is **extremely common** in real data analysis — one `agg()` call gives you a whole summary table.


## PART 5 — `transform()`

Unlike `groupby().mean()` (which returns one row per group), `transform()` returns **the same number of rows as the original DataFrame**.

It broadcasts a per-group result back to every row in that group.


In [52]:
df["Department_Mean"] = (
    df.groupby("Department")["Math"].transform("mean")
)
print(df[["Name", "Department", "Math", "Department_Mean"]])

      Name Department  Math  Department_Mean
0    Alice         CS    80        81.666667
1      Bob         IT    70        82.500000
2  Charlie         CS    90        81.666667
3    David         SE    60        74.000000
4      Eve         IT    95        82.500000
5    Frank         CS    75        81.666667
6    Grace         SE    88        74.000000


Every row is preserved (notice Alice, Charlie, Frank all share `81.67` for CS).


This is very important for Machine Learning — you often want a per-group statistic **aligned to each row**.


### Why not `agg()` here?

`agg()` returns one row per group. `transform()` returns one row per **original row**. To attach a group summary back to the original rows, use `transform()`.


## PART 6 — `filter()`

`filter()` keeps **entire groups** that satisfy a condition (rows are not filtered individually).


In [53]:
# Departments with at least 3 students
big_depts = df.groupby("Department").filter(lambda x: len(x) >= 3)
print(big_depts)

      Name Department  Math  Python  AI   Department_Name   Row_Mean  \
0    Alice         CS    80      90  88  Computer Science  86.000000   
2  Charlie         CS    90      95  92  Computer Science  92.333333   
5    Frank         CS    75      80  78  Computer Science  77.666667   

   Math_Squared  Pass Grade    Average  Department_Mean  
0          6400  Pass     A  86.000000        81.666667  
2          8100  Pass     A  92.333333        81.666667  
5          5625  Pass     B  77.666667        81.666667  


`SE` (2 students) is dropped — `CS` and `IT` (3 students each) survive.


## PART 7 — Pivot Tables

### What is a pivot table?

A pivot table summarizes data like `groupby`, but produces a **nice 2D grid** (perfect for spreadsheets / presentations).

**Signature**
```python
pd.pivot_table(df, values=..., index=..., columns=..., aggfunc=...)
```

### Single value, single index


In [54]:
# Self-contained reset so the pivot / crosstab examples below work even
# if the notebook was not executed from the top (Grade is normally created
# in Part 2, and df may have been overwritten by later cells).
df = pd.DataFrame(students)
df["Grade"] = df["Python"].apply(
    lambda s: "A" if s >= 90 else "B" if s >= 80 else "C" if s >= 70 else "D"
)

In [75]:
print(pd.pivot_table(
    df,
    values="Math",
    index="Department",
    aggfunc="mean"
))

                 Math
Department           
CS          81.666667
IT          83.333333
SE          74.000000


### Multiple values


In [76]:
print(pd.pivot_table(
    df,
    values=["Math", "Python"],
    index="Department",
    aggfunc="mean"
))

                 Math     Python
Department                      
CS          81.666667  88.333333
IT          83.333333  92.333333
SE          74.000000  77.500000


### Multiple indexes


In [57]:
print(pd.pivot_table(
    df,
    values="Math",
    index=["Department", "Grade"],
    aggfunc="mean"
))

                  Math
Department Grade      
CS         A      85.0
           B      75.0
IT         A      95.0
           B      70.0
SE         B      88.0
           C      60.0


### Pivot table with `columns=`


In [58]:
print(pd.pivot_table(
    df,
    values="Math",
    index="Department",
    columns="Grade",
    aggfunc="mean"
))

Grade          A     B     C
Department                  
CS          85.0  75.0   NaN
IT          95.0  70.0   NaN
SE           NaN  88.0  60.0


## PART 8 — Crosstab

`pd.crosstab()` **counts combinations** of two (or more) categorical columns.

### Department vs Grade


In [59]:
print(pd.crosstab(df["Department"], df["Grade"]))

Grade       A  B  C
Department         
CS          2  1  0
IT          1  1  0
SE          0  1  1


This is extremely useful for **classification** problems — it is a confusion-matrix-style count of categories.


## PART 9 — Feature Engineering

Feature engineering = creating new, useful columns from existing data.

This is a core **Data Scientist skill**.

### Average score


In [60]:
df["Average"] = df[["Math", "Python", "AI"]].mean(axis=1)
print(df[["Name", "Average"]])

      Name    Average
0    Alice  86.000000
1      Bob  76.666667
2  Charlie  92.333333
3    David  65.000000
4      Eve  97.666667
5    Frank  77.666667
6    Grace  86.666667


### Best subject (`idxmax`)

`idxmax(axis=1)` returns the **column name** of the largest value **per row**.


In [61]:
df["Best_Subject"] = df[["Math", "Python", "AI"]].idxmax(axis=1)
print(df[["Name", "Math", "Python", "AI", "Best_Subject"]])

      Name  Math  Python  AI Best_Subject
0    Alice    80      90  88       Python
1      Bob    70      85  75       Python
2  Charlie    90      95  92       Python
3    David    60      70  65       Python
4      Eve    95     100  98       Python
5    Frank    75      80  78       Python
6    Grace    88      85  87         Math


### Weakest subject (`idxmin`)


In [62]:
df["Weakest_Subject"] = df[["Math", "Python", "AI"]].idxmin(axis=1)
print(df[["Name", "Best_Subject", "Weakest_Subject"]])

      Name Best_Subject Weakest_Subject
0    Alice       Python            Math
1      Bob       Python            Math
2  Charlie       Python            Math
3    David       Python            Math
4      Eve       Python            Math
5    Frank       Python            Math
6    Grace         Math          Python


### Performance label (lambda with multiple conditions)


In [63]:
df["Performance"] = df["Average"].apply(
    lambda x:
        "Excellent" if x >= 90
        else "Good" if x >= 80
        else "Needs Improvement"
)
print(df[["Name", "Average", "Performance"]])

      Name    Average        Performance
0    Alice  86.000000               Good
1      Bob  76.666667  Needs Improvement
2  Charlie  92.333333          Excellent
3    David  65.000000  Needs Improvement
4      Eve  97.666667          Excellent
5    Frank  77.666667  Needs Improvement
6    Grace  86.666667               Good


### Department average (`transform`)


In [64]:
df["Department_Average"] = (
    df.groupby("Department")["Average"].transform("mean")
)
print(df[["Name", "Department", "Average", "Department_Average"]])

      Name Department    Average  Department_Average
0    Alice         CS  86.000000           85.333333
1      Bob         IT  76.666667           87.166667
2  Charlie         CS  92.333333           85.333333
3    David         SE  65.000000           75.833333
4      Eve         IT  97.666667           87.166667
5    Frank         CS  77.666667           85.333333
6    Grace         SE  86.666667           75.833333


### Difference from department average

A real **ML feature** — how far each student is from their department's average.


In [72]:
df["Difference"] = df["Average"] - df["Department_Average"]
print(df[["Name", "Department", "Average", "Department_Average", "Difference"]])

      Name Department    Average  Department_Average  Difference
0    Alice         CS  86.000000           85.333333    0.666667
1      Bob         IT  76.666667           87.777778  -11.111111
2  Charlie         CS  92.333333           85.333333    7.000000
3    David         SE  65.000000           75.833333  -10.833333
4      Eve         IT  97.666667           87.777778    9.888889
5    Frank         CS  77.666667           85.333333   -7.666667
6    Grace         SE  86.666667           75.833333   10.833333
7    Henry         IT  89.000000           87.777778    1.222222


### Common workflow

The whole pipeline end to end:


In [73]:
summary = (
    df.groupby("Department")
      .agg({
          "Average": "mean",
          "Math": "max",
          "Python": "max"
      })
)
print(summary)

              Average  Math  Python
Department                         
CS          85.333333    90      95
IT          87.777778    95     100
SE          75.833333    88      85


### Putting it together — reload from CSV workflow


In practice you often start from a CSV:


In [74]:
# Simulate the real workflow: load from CSV, engineer features, summarize
df.head()

,Name,Department,Math,Python,AI,Average,Grade,Performance,Best_Subject,Weakest_Subject,Total,Department_Average,Difference_From_Department,Difference
0,Alice,CS,80,90,88,86.000000,B,Good,Python,Math,258,85.333333,0.666667,0.666667
1,Bob,IT,70,85,75,76.666667,C,Needs Improvement,Python,Math,230,87.777778,-11.111111,-11.111111
2,Charlie,CS,90,95,92,92.333333,A,Excellent,Python,Math,277,85.333333,7.000000,7.000000
3,David,SE,60,70,65,65.000000,D,Needs Improvement,Python,Math,195,75.833333,-10.833333,-10.833333
4,Eve,IT,95,100,98,97.666667,A,Excellent,Python,Math,293,87.777778,9.888889,9.888889


## Cheat Sheet — Day 4

```
map()       df["col"].map(dict_or_func)      # transform one Series
replace()   s.replace(dict) / df.replace(dict)  # Series or DataFrame
apply()     df.apply(func, axis=0)            # apply a function
assign()    df.assign(new=df["col"]*2)        # chainable column creation
lambda      lambda x: x*2                     # short anonymous function

GROUPBY
groupby()   df.groupby("col")                 # lazy — splits into groups
.agg()      .agg({"col": "mean"})             # one summary per group
.transform() .transform("mean")               # broadcast stats back to rows
.filter()   .filter(lambda g: len(g) > 2)     # keep whole groups

PIVOT / COUNT
pivot_table()  pd.pivot_table(df, values=, index=, columns=, aggfunc=)
crosstab()     pd.crosstab(df["a"], df["b"])   # count combinations

FEATURE ENGINEERING
.mean(axis=1)  row-wise mean
.idxmax(axis=1)  column name of max per row
.apply(func)    map a function to values
```


## Day 4 Knowledge Check

Answer these questions in your own words before continuing:

---

**1.** What is the difference between `map()` and `replace()`?

**2.** What is the difference between `apply()` and `map()`?

**3.** What does `groupby()` return before an aggregation?

**4.** What is the difference between `count()` and `size()`?

**5.** Why is `transform()` different from `agg()`?

**6.** When would you use a pivot table instead of `groupby()`?

**7.** What does `idxmax()` return?

**8.** What does `assign()` do?

**9.** Why are lambda functions useful?

**10.** What is feature engineering?

---

**Your answers:**

*(Write your answers below each question.)*


## 💻 Coding Challenge

Use the dataset below and complete the tasks.


In [68]:
employees = {
    "Name": ["John", "Mary", "Peter", "Jane", "Mike", "Sara", "Tom"],
    "Department": ["IT", "HR", "IT", "Finance", "IT", "HR", "Finance"],
    "Salary": [5000, 4000, 4500, 6000, 7000, 4800, 8000],
    "Experience": [2, 5, 1, 4, 8, 3, 12]
}

df = pd.DataFrame(employees)
print("EMPLOYEE DATASET:")
print(df)

EMPLOYEE DATASET:
    Name Department  Salary  Experience
0   John         IT    5000           2
1   Mary         HR    4000           5
2  Peter         IT    4500           1
3   Jane    Finance    6000           4
4   Mike         IT    7000           8
5   Sara         HR    4800           3
6    Tom    Finance    8000          12


**Tasks:**

1. Create a `Salary_Level` column:
   - `Salary >= 7000` → **High**
   - `5000–6999` → **Medium**
   - Below `5000` → **Low**
2. Add an `Experience_Level` column:
   - `>= 10` → **Expert**
   - `>= 5` → **Senior**
   - `>= 2` → **Mid**
   - Otherwise → **Junior**
3. Calculate average salary for each department.
4. Calculate maximum salary for each department.
5. Add `Department_Average_Salary` using `transform()`.
6. Create `Difference_From_Department`.
7. Create a pivot table of average salary by department.
8. Count `Salary_Level` by Department using `crosstab()`.
9. Find the highest-paid employee in each department.
10. Display the final DataFrame.


In [ ]:
# Your code here:

# 1. Salary_Level
def salary_level(s):
    if s >= 7000:
        return "High"
    elif s >= 5000:
        return "Medium"
    return "Low"

df["Salary_Level"] = df["Salary"].apply(salary_level)

# 2. Experience_Level
def experience_level(e):
    if e >= 10:
        return "Expert"
    elif e >= 5:
        return "Senior"
    elif e >= 2:
        return "Mid"
    return "Junior"

df["Experience_Level"] = df["Experience"].apply(experience_level)

# 3. Average salary per department
print("Average salary per department:")
print(df.groupby("Department")["Salary"].mean())
print()

# 4. Maximum salary per department
print("Maximum salary per department:")
print(df.groupby("Department")["Salary"].max())
print()

# 5. Department_Average_Salary using transform()
df["Department_Average_Salary"] = df.groupby("Department")["Salary"].transform("mean")

# 6. Difference_From_Department
df["Difference_From_Department"] = df["Salary"] - df["Department_Average_Salary"]

# 7. Pivot table of average salary by department
print("Pivot table — average salary by department:")
print(pd.pivot_table(df, values="Salary", index="Department", aggfunc="mean"))
print()

# 8. Crosstab: Salary_Level by Department
print("Crosstab — Salary_Level by Department:")
print(pd.crosstab(df["Department"], df["Salary_Level"]))
print()

# 9. Highest-paid employee in each department
idx = df.groupby("Department")["Salary"].idxmax()
print("Highest-paid employee in each department:")
print(df.loc[idx, ["Name", "Department", "Salary"]])
print()

# 10. Display final DataFrame
print("FINAL DATAFRAME:")
print(df)

Average salary per department:
Department
Finance    7000.0
HR         4400.0
IT         5500.0
Name: Salary, dtype: float64

Maximum salary per department:
Department
Finance    8000
HR         4800
IT         7000
Name: Salary, dtype: int64

Pivot table — average salary by department:
            Salary
Department        
Finance     7000.0
HR          4400.0
IT          5500.0

Crosstab — Salary_Level by Department:
Salary_Level  High  Low  Medium
Department                     
Finance          1    0       1
HR               0    2       0
IT               1    1       1

Highest-paid employee in each department:
   Name Department  Salary
6   Tom    Finance    8000
5  Sara         HR    4800
4  Mike         IT    7000

FINAL DATAFRAME:
    Name Department  Salary  Experience Salary_Level Experience_Level  \
0   John         IT    5000           2       Medium              Mid   
1   Mary         HR    4000           5          Low           Senior   
2  Peter         IT    4500  

## 🚀 Mini Project — Student Performance Analytics

Apply everything you've learned in a complete end-to-end workflow.


In [77]:
students = {
    "Name": [
        "Alice",
        "Bob",
        "Charlie",
        "David",
        "Eve",
        "Frank",
        "Grace",
        "Henry"
    ],
    "Department": [
        "CS",
        "IT",
        "CS",
        "SE",
        "IT",
        "CS",
        "SE",
        "IT"
    ],
    "Math":    [80, 70, 90, 60, 95, 75, 88, 85],
    "Python":  [90, 85, 95, 70, 100, 80, 85, 92],
    "AI":      [88, 75, 92, 65, 98, 78, 87, 90]
}

df = pd.DataFrame(students)
print(df)

      Name Department  Math  Python  AI
0    Alice         CS    80      90  88
1      Bob         IT    70      85  75
2  Charlie         CS    90      95  92
3    David         SE    60      70  65
4      Eve         IT    95     100  98
5    Frank         CS    75      80  78
6    Grace         SE    88      85  87
7    Henry         IT    85      92  90


Complete all tasks:

1. Create an `Average` column.
2. Create a `Grade` column using: **A ≥ 90**, **B ≥ 80**, **C ≥ 70**, **D < 70**.
3. Create a `Performance` column: **Excellent**, **Good**, **Needs Improvement**.
4. Create `Best_Subject`.
5. Create `Weakest_Subject`.
6. Create `Total` score.
7. Calculate department average using `transform()`.
8. Create `Difference_From_Department`.
9. Group by department and calculate: **Mean Average**, **Max Math**, **Max Python**, **Min AI**.
10. Create a pivot table showing average scores by department.
11. Create a crosstab of Department vs Grade.
12. Find the top-performing student in each department.
13. Sort students by Average (highest first).
14. Save as `student_performance_report.csv`.
15. Reload the CSV and display it.


In [78]:
# Your mini project code here:

# 1. Average column
df["Average"] = df[["Math", "Python", "AI"]].mean(axis=1)

# 2. Grade column
def grade(avg):
    if avg >= 90:
        return "A"
    elif avg >= 80:
        return "B"
    elif avg >= 70:
        return "C"
    return "D"

df["Grade"] = df["Average"].apply(grade)

# 3. Performance column
df["Performance"] = df["Average"].apply(
    lambda x: "Excellent" if x >= 90 else ("Good" if x >= 80 else "Needs Improvement")
)

# 4. Best_Subject
df["Best_Subject"] = df[["Math", "Python", "AI"]].idxmax(axis=1)

# 5. Weakest_Subject
df["Weakest_Subject"] = df[["Math", "Python", "AI"]].idxmin(axis=1)

# 6. Total score
df["Total"] = df[["Math", "Python", "AI"]].sum(axis=1)

# 7. Department average using transform()
df["Department_Average"] = df.groupby("Department")["Average"].transform("mean")

# 8. Difference from department average
df["Difference_From_Department"] = df["Average"] - df["Department_Average"]

# 9. Group by department summary
dept_summary = df.groupby("Department").agg({
    "Average": "mean",
    "Math": "max",
    "Python": "max",
    "AI": "min"
})
print("=== Department Summary ===")
print(dept_summary)
print()

# 10. Pivot table: average scores by department
print("=== Pivot Table: Average Scores by Department ===")
print(pd.pivot_table(df, values=["Math", "Python", "AI"], index="Department", aggfunc="mean"))
print()

# 11. Crosstab: Department vs Grade
print("=== Crosstab: Department vs Grade ===")
print(pd.crosstab(df["Department"], df["Grade"]))
print()

# 12. Top-performing student in each department
top_idx = df.groupby("Department")["Average"].idxmax()
print("=== Top-performing student per department ===")
print(df.loc[top_idx, ["Name", "Department", "Average"]])
print()

# 13. Sort students by Average (highest first)
df_sorted = df.sort_values(by="Average", ascending=False)
print("=== Sorted by Average (highest first) ===")
print(df_sorted[["Name", "Department", "Average", "Grade"]])
print()

# 14. Save as CSV
df.to_csv("student_performance_report.csv", index=False)
print("Saved student_performance_report.csv")
print()

# 15. Reload the CSV and display it
print("=== Reloaded from CSV ===")
loaded = pd.read_csv("student_performance_report.csv")
print(loaded)

=== Department Summary ===
              Average  Math  Python  AI
Department                             
CS          85.333333    90      95  78
IT          87.777778    95     100  75
SE          75.833333    88      85  65

=== Pivot Table: Average Scores by Department ===
                   AI       Math     Python
Department                                 
CS          86.000000  81.666667  88.333333
IT          87.666667  83.333333  92.333333
SE          76.000000  74.000000  77.500000

=== Crosstab: Department vs Grade ===
Grade       A  B  C  D
Department            
CS          1  1  1  0
IT          1  1  1  0
SE          0  1  0  1

=== Top-performing student per department ===
      Name Department    Average
2  Charlie         CS  92.333333
4      Eve         IT  97.666667
6    Grace         SE  86.666667

=== Sorted by Average (highest first) ===
      Name Department    Average Grade
4      Eve         IT  97.666667     A
2  Charlie         CS  92.333333     A
7    Henr

## Day 4 Summary

By the end of this lesson, you should be comfortable with:

| Tool | Purpose |
|------|---------|
| `map()` | Transform one Series |
| `replace()` | Replace values in a Series or DataFrame |
| `apply()` | Apply a function to a Series / DataFrame |
| `lambda` | Short anonymous functions |
| `assign()` | Chainable column creation |
| `groupby()` | Split data into groups |
| `agg()` | One summary per group |
| `transform()` | Broadcast group stats back to rows |
| `filter()` | Keep whole groups matching a condition |
| `pivot_table()` | 2D summary table |
| `crosstab()` | Count category combinations |
| `idxmax()` / `idxmin()` | Find column name of max/min per row |

These are among the most valuable Pandas skills because they let you turn raw data into meaningful insights. They also appear frequently in data science interviews and real-world analytics work.

